In [1]:
import numpy as np
import torch
import os

In [121]:
B, T = 4, 2048
buf = torch.tensor(tokens[:B*T+1])
x = buf[:-1].view(B, T)
y = buf[1:].view(B, T)

In [20]:
tokens.size 

12278694080

In [32]:
import warnings
warnings.filterwarnings("ignore")

In [30]:
class DataLoader:
    def __init__(self, path, B, T):
        self.B = B
        self.T = T

        self.tokens = np.memmap(path, dtype=np.uint16, mode="r")
        print(f"loaded {self.tokens.size} tokens")
        print(f"1 epoch {self.tokens.size // (B * T)} batches")

        #state
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = torch.from_numpy(self.tokens[self.current_position : self.current_position + B*T+1])
        x = buf[:-1].view(B, T) #inputs
        y = buf[1:].view(B, T) #targets

        #advance position in tokens
        self.current_position += B * T

        #reset when you reach end of dataset
        if self.current_position + (B * T + 1) > self.tokens.size:
            self.current_position = 0

        return x, y
        

In [124]:
train_loader = DataLoader("final_data/train.bin", B, T)


loaded 12278694080 tokens
1 epoch 1498864 batches


In [125]:
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("tokenizer.json")

In [ ]:
x, y = train_loader.next_batch()
print("".join(tokenizer.decode_batch(x.tolist(), skip_special_tokens=False)))

In [170]:
from model import get_model
from dataclasses import dataclass 

In [172]:

@dataclass
class DecoderConfig:
    seq_len: int = 4096
    vocab_size: int =50304
    n_blocks: int = 36
    n_head: int = 16
    n_embd: int = 768
    base: int = 10000

m = get_model(DecoderConfig())
m


KeyError: "attribute 'theta' already exists"